# 08 · Exploratory analysis for modelling

Prepares the poverty-vulnerability and VUP coverage-gap models:
- who is poor and who is distressed;
- how features relate to each other;
- how VUP coverage is spread among high-need households;
- which variables would leak the targets.

| | |
|---|---|
| **Inputs** | `data/processed/household_master.parquet`, `data/processed/household_cpi_shock.parquet` |
| **Outputs** | `data/processed/model_splits.parquet` (the cluster split used by notebooks 09-11), `reports/figures/modeling_eda/*.png`, `reports/modeling_eda_summary.md` |
| **Code** | `src/models/common.py`, `src/models/plots.py` |
| **Kernel** | `Umuseke (.venv)` (project virtual environment with scikit-learn, LightGBM, XGBoost, SHAP) |

**Holdout first.** The split is made before any analysis:
- **Final holdout:** one survey cluster in five, chosen within each stratum. It is never looked at here.
- **Development set:** every chart and table below uses only the development set.

Poverty rates are **person-level** (`pop_wt`). Distress scores and VUP coverage are **household-level** (`weight`). Intervals are 95% design-based intervals (Taylor linearisation with strata and clusters).

In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "models" / "common.py").exists())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import roc_auc_score

from src.models import common as cm
from src.models import plots as pl

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:,.3f}".format)
FIG = ROOT / "reports" / "figures" / "modeling_eda"
FIG.mkdir(parents=True, exist_ok=True)

## 1. Cluster-aware split

In [ ]:
df = cm.load_frame()
splits = cm.make_splits(df)
splits.to_parquet(cm.SPLITS_PATH, index=False)
d = df.merge(splits, on=["hhid", "clust"])
bal = d.groupby("split").apply(lambda g: pd.Series({
    "households": len(g), "clusters": g["clust"].nunique(), "strata": g["strata_id"].nunique(),
    "weighted households": g["weight"].sum(), "poverty rate (people)": np.average(g["poor"], weights=g["pop_wt"]),
    "VUP coverage (households)": np.average(g["vup_any"], weights=g["weight"]),
    "urban share (households)": np.average(g["urban"], weights=g["weight"])}))
display(bal)
print("Development households per CV fold:", d[d["split"].eq("development")]["fold"].value_counts().sort_index().to_dict())
small = d.groupby("strata_id")["clust"].nunique()
print(f"Strata with no holdout cluster (2-3 clusters each): {d.groupby('strata_id')['split'].apply(lambda s: (s == 'holdout').sum() == 0).sum()}")
dev = d[d["split"].eq("development")].reset_index(drop=True)

The two parts have similar poverty, VUP and urban shares, so the holdout is representative. From here on, only `dev` (the development set) is used.

## 2. Poverty and distress by group

In [ ]:
dev["area_name"] = dev["area"].str.capitalize()
dev["income_source"] = dev["main_livelihood_pf"].map({"own_farming": "Own farming", "farm_wage": "Farm wage work",
    "nonfarm_wage": "Non-farm wage work", "nonfarm_self_employment": "Non-farm self-employment", "no_employed_member": "No employed member"})
dev["head_education"] = dev["head_education_level"].map({"none": "None", "primary": "Primary", "post_primary_vocational": "Post-primary vocational",
    "lower_secondary": "Lower secondary", "upper_secondary": "Upper secondary", "tertiary": "Tertiary"})
dev["household_size"] = pd.cut(dev["hh_size"], [0, 1, 3, 5, 7, 99], labels=["1 person", "2-3", "4-5", "6-7", "8+"]).astype(str)
dev["dependency"] = np.select([dev["no_working_age_member"].eq(1), dev["dependency_ratio"].eq(0), dev["dependency_ratio"].le(1), dev["dependency_ratio"].le(2)],
                              ["No member 15-64", "0", "0-1", "1-2"], "Above 2")
dev["head_sex"] = np.where(dev["head_female"].eq(1), "Female head", "Male head")
dev["key_assets"] = pd.cut(dev["n_key_assets"], [-1, 0, 1, 2, 3, 99], labels=["0", "1", "2", "3", "4+"]).astype(str)
dev["vup_status"] = np.where(dev["vup_any"].eq(1), "Any VUP component", "No VUP")
DIMS = {
    "area": ("Urban / rural", "area_name", ["Urban", "Rural"]),
    "income_source": ("Main income source (employment-based)", "income_source",
                      ["Own farming", "Farm wage work", "Non-farm wage work", "Non-farm self-employment", "No employed member"]),
    "education": ("Head's education", "head_education", ["None", "Primary", "Post-primary vocational", "Lower secondary", "Upper secondary", "Tertiary"]),
    "household_size": ("Household size", "household_size", ["1 person", "2-3", "4-5", "6-7", "8+"]),
    "dependency": ("Dependency ratio", "dependency", ["0", "0-1", "1-2", "Above 2", "No member 15-64"]),
    "head_sex": ("Head's sex", "head_sex", ["Male head", "Female head"]),
    "assets": ("Number of key assets (radio, TV, phone, bicycle, motorcycle, car, fridge, computer)", "key_assets", ["0", "1", "2", "3", "4+"]),
    "vup": ("VUP participation", "vup_status", ["No VUP", "Any VUP component"]),
}
pov_all = cm.weighted_rate_table(dev.assign(all="Rwanda"), "poor", "all")["mean"].iloc[0]
dis_all = cm.weighted_rate_table(dev.assign(all="Rwanda"), "distress_score", "all", weight="weight")["mean"].iloc[0]
tables = {}
for key, (label, col, order) in DIMS.items():
    tables[key] = (cm.weighted_rate_table(dev, "poor", col, "pop_wt", order),
                   cm.weighted_rate_table(dev, "distress_score", col, "weight", order))
print(f"Development set: poverty {100 * pov_all:.1f}% of people; mean distress score {dis_all:.1f}")

In [ ]:
def two_panel(key, title, path):
    label, col, order = DIMS[key]
    pov, dis = tables[key]
    n = len(order)
    fig, (a1, a2) = pl.figure(title, f"{label}. Dots: estimate; lines: 95% confidence interval. Development set", width=11, height=1.9 + 0.42 * n, ncols=2)
    pl.dot_ci(a1, pov, 100, pl.SERIES[0], reference=pov_all)
    pl.dot_ci(a2, dis.assign(mean=dis["mean"] / 100, ci_low=dis["ci_low"] / 100, ci_high=dis["ci_high"] / 100), 100, pl.SERIES[1], reference=dis_all / 100)
    a2.set_yticks(a2.get_yticks(), [""] * n)
    a1.set_title("Poor, % of people (pop_wt)", fontsize=10, color=pl.INK["secondary"], loc="left")
    a2.set_title("Mean distress score, 0-100 (weight)", fontsize=10, color=pl.INK["secondary"], loc="left")
    a1.set_xlim(0, max(100 * pov["ci_high"].max() * 1.15, 10))
    a2.set_xlim(min(dis["ci_low"].min() * 0.9, 40), dis["ci_high"].max() * 1.08)
    return pl.save(fig, path, top=1 - 1.25 / (1.9 + 0.42 * n), bottom=0.5 / (1.9 + 0.42 * n), left=0.24, right=0.97)

charts = {}
for key, title in [("area", "Rural households are poorer and more distressed"),
                   ("income_source", "Farm wage workers have the highest poverty and distress"),
                   ("education", "Poverty falls steeply with the head's education"),
                   ("household_size", "Larger households are poorer"),
                   ("dependency", "More dependants per working-age adult, more poverty"),
                   ("head_sex", "Female-headed households: poverty and distress"),
                   ("assets", "Poverty falls steeply with key assets"),
                   ("vup", "VUP participants are poorer, but less distressed once VUP is counted")]:
    charts[key] = two_panel(key, title, FIG / f"poverty_distress_by_{key}.png")
for key in ["income_source", "assets"]:
    display(Image(str(charts[key])))

In [ ]:
summary_rows = []
for key, (pov, dis) in tables.items():
    for g in pov.index:
        summary_rows.append({"dimension": key, "group": g, "households": int(pov.loc[g, "households"]),
                             "poor % people": 100 * pov.loc[g, "mean"], "poor CI": f"{100 * pov.loc[g, 'ci_low']:.1f}-{100 * pov.loc[g, 'ci_high']:.1f}",
                             "distress mean": dis.loc[g, "mean"], "distress CI": f"{dis.loc[g, 'ci_low']:.1f}-{dis.loc[g, 'ci_high']:.1f}"})
group_summary = pd.DataFrame(summary_rows)
group_summary

### Districts

District estimates rest on about 390 development-set households each, so their intervals are wide. Districts whose intervals overlap should not be ranked against each other.

In [ ]:
order = cm.weighted_rate_table(dev, "poor", "district_name", "pop_wt").sort_values("mean", ascending=False).index.tolist()
pov_d = cm.weighted_rate_table(dev, "poor", "district_name", "pop_wt", order)
dis_d = cm.weighted_rate_table(dev, "distress_score", "district_name", "weight", order)
n = len(order)
fig, (a1, a2) = pl.figure("Poverty and distress by district: wide, overlapping intervals",
                          "Sorted by poverty rate. Dots: estimate; lines: 95% confidence interval. Development set (about 390 households per district)",
                          width=11, height=1.9 + 0.3 * n, ncols=2)
pl.dot_ci(a1, pov_d, 100, pl.SERIES[0], reference=pov_all)
pl.dot_ci(a2, dis_d.assign(mean=dis_d["mean"] / 100, ci_low=dis_d["ci_low"] / 100, ci_high=dis_d["ci_high"] / 100), 100, pl.SERIES[1], reference=dis_all / 100)
a2.set_yticks(a2.get_yticks(), [""] * n)
a1.set_title("Poor, % of people", fontsize=10, color=pl.INK["secondary"], loc="left")
a2.set_title("Mean distress score", fontsize=10, color=pl.INK["secondary"], loc="left")
a1.set_xlim(0, 100 * pov_d["ci_high"].max() * 1.12)
a2.set_xlim(dis_d["ci_low"].min() * 0.92, dis_d["ci_high"].max() * 1.06)
path = pl.save(fig, FIG / "poverty_distress_by_district.png", top=1 - 1.25 / (1.9 + 0.3 * n), bottom=0.5 / (1.9 + 0.3 * n), left=0.2, right=0.97)
display(Image(str(path)))
overlap = (pov_d["ci_low"].max() < pov_d["ci_high"].min())
print(f"Poverty range across districts: {100 * pov_d['mean'].min():.1f}% to {100 * pov_d['mean'].max():.1f}%; median CI width {100 * (pov_d['ci_high'] - pov_d['ci_low']).median():.1f} pp")
top, bottom = pov_d.index[0], pov_d.index[-1]
print(f"Districts whose CI overlaps the highest district ({top}): {int((pov_d['ci_high'] >= pov_d.loc[top, 'ci_low']).sum()) - 1}")

## 3. Correlations and multicollinearity

The candidate features are the full poverty-model feature set from `src/models/common.py`. Leakage variables are already removed (section 5).

In [ ]:
spec = cm.FeatureSpec("full", list(cm.GROUPS))
cm.assert_no_leakage(spec, "poverty")
num = dev[spec.numeric + spec.money].astype(float)
num[spec.money] = np.log1p(num[spec.money])
num = num.fillna(num.median())
num = num.loc[:, num.std() > 0]
corr = num.corr()
pairs = (corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack().rename("r").reset_index()
         .rename(columns={"level_0": "feature_1", "level_1": "feature_2"}))
high_pairs = pairs[pairs["r"].abs() >= 0.8].sort_values("r", key=np.abs, ascending=False)
print(f"{num.shape[1]} numeric features; {len(high_pairs)} pairs with |r| >= 0.8")
display(high_pairs.head(25))

In [ ]:
inv = np.linalg.pinv(corr.to_numpy())
vif = pd.Series(np.diag(inv), index=corr.columns, name="VIF").sort_values(ascending=False)
print(f"Features with VIF > 10: {int((vif > 10).sum())}; VIF > 5: {int((vif > 5).sum())}")
display(vif.head(25).to_frame())

In [ ]:
key_feats = ["hh_size", "n_children_0_14", "dependency_ratio", "head_age", "head_female", "head_education_code", "max_education_code",
             "share_employed_in_agriculture_pf", "share_employed_informal_proxy_pf", "any_wage_worker_pf", "n_key_assets",
             "tropical_livestock_units", "electricity_any", "improved_floor", "persons_per_sleeping_room", "minutes_to_health_centre",
             "any_account", "share_food", "share_meals_away", "share_food_own_produced", "price_change_pct", "food_shock_pp",
             "distress_score_model", "urban"]
c = num[key_feats].corr()
import matplotlib.pyplot as plt
fig, (ax,) = pl.figure("Correlations among key candidate features", "Pearson correlation, development set (missing values set to the median)", width=10, height=9)
im = ax.imshow(c, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(key_feats)), key_feats, rotation=90, fontsize=7.5)
ax.set_yticks(range(len(key_feats)), key_feats, fontsize=7.5)
ax.grid(False)
for i in range(len(key_feats)):
    for j in range(len(key_feats)):
        if i != j and abs(c.iloc[i, j]) >= 0.5:
            ax.text(j, i, f"{c.iloc[i, j]:.1f}", ha="center", va="center", fontsize=6, color="white" if abs(c.iloc[i, j]) > 0.7 else pl.INK["primary"])
cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cb.ax.tick_params(labelsize=8)
path = pl.save(fig, FIG / "correlation_key_features.png", top=0.9, bottom=0.2, left=0.25, right=0.95)
display(Image(str(path)))

**Implications for modelling.**
- **Collinear groups are expected:** household size and member counts, livelihood counts and shares, durables and the asset count.
- **They do not bias predictions,** but they make logistic-regression coefficients unstable and spread tree importance across substitutes.
- **The models keep all features** (no tuning at this stage). Interpretation in notebook 11 reports SHAP importance aggregated by feature group, and treats single coefficients with care.

## 4. Single-feature screen against poverty

The weighted AUC of each feature on its own (shown as max(AUC, 1 − AUC), so direction does not matter). A legitimate feature with an AUC close to 1 would suggest leakage; the known leakage variables are shown for comparison.

In [ ]:
def uni_auc(x, y, w):
    ok = x.notna()
    a = roc_auc_score(y[ok], x[ok], sample_weight=w[ok])
    return max(a, 1 - a)

screen = pd.Series({f: uni_auc(dev[f].astype(float), dev["poor"], dev["weight"]) for f in spec.numeric + spec.money}).sort_values(ascending=False)
leak_check = pd.Series({f: uni_auc(dev[f].astype(float), dev["poor"], dev["weight"]) for f in
                        ["welfare_ratio", "cons_real_ae_jan2024", "c_poverty", "distress_score", "distress_score_excl_vup", "exp9", "extra_cost_same_basket_rwf_per_ae", "hh_price_index"]})
print("Known leakage variables (excluded):")
display(leak_check.to_frame("AUC"))
print("Top 20 allowed features:")
display(screen.head(20).to_frame("AUC"))
top20 = screen.head(20)[::-1]
fig, (ax,) = pl.figure("Strongest single predictors of poverty among allowed features",
                       "Weighted AUC of each feature alone, max(AUC, 1 - AUC); 0.5 = no information. Development set", width=10, height=6.5)
ax.barh(range(len(top20)), top20 - 0.5, left=0.5, color=pl.SERIES[0], height=0.65)
ax.set_yticks(range(len(top20)), top20.index, fontsize=8.5)
ax.set_xlim(0.5, 1)
for i, v in enumerate(top20):
    ax.text(v + 0.005, i, f"{v:.2f}", va="center", fontsize=8)
path = pl.save(fig, FIG / "univariate_auc_poverty.png", top=0.86, bottom=0.1, left=0.32)
display(Image(str(path)))
print(f"Highest allowed single-feature AUC: {screen.iloc[0]:.3f} ({screen.index[0]}); features above 0.85: {list(screen[screen > 0.85].index)}")

## 5. Leakage

Variables that would let a model "see" the answer are kept out by `cm.assert_no_leakage`. These are the rules:

In [ ]:
leak = pd.DataFrame([{"model": t, "variable or pattern": v, "reason": r} for t, d_ in cm.LEAKAGE.items() for v, r in d_.items()])
display(leak)

In [ ]:
# Hidden VUP signals found in non-VUP modules
vjob = dev["n_vup_sector_jobs"].gt(0)
pw = dev["vup_classic_pw"].eq(1) | dev["vup_expanded_pw"].eq(1)
print(f"Households with a main job in the 'VUP' institutional sector: {int(vjob.sum())}; of these in VUP Public Works: {int((vjob & pw).sum())}")
print(f"Households with a VUP Financial Services loan in the credit module: {int(dev['has_vup_fs_loan'].sum())}; "
      f"of these flagged as VUP FS participants: {int((dev['has_vup_fs_loan'].eq(1) & dev['vup_financial_services'].eq(1)).sum())}")
a1 = roc_auc_score(dev["vup_any"], dev["n_nonfarm_wage"], sample_weight=dev["weight"])
a2 = roc_auc_score(dev["vup_any"], dev["n_nonfarm_wage_pf"], sample_weight=dev["weight"])
print(f"AUC of non-farm wage workers for VUP participation: {a1:.3f} with VUP jobs counted, {a2:.3f} program-free")

**Program-free features.** Employment and loan features are rebuilt without VUP-sector jobs and VUP loans (columns ending `_pf` and `_excl_vup`). Both models use these versions:
- In the VUP model, the original versions would leak the target.
- In the poverty model, they would make the model learn current program targeting rather than poverty.

For the same reason, the poverty model leaves out every program and public-transfer variable (VUP, Girinka, S9E public support).

**The distress score as a feature.** The published distress score contains the poverty component (`c_poverty`) and the VUP component. The poverty model therefore uses `distress_score_model`: inflation, food share, income instability and the savings/credit gap, built program-free. The VUP model may use poverty information but nothing about VUP.

**Consumption-derived features.** Expenditure shares, the CPI shock and the distress score all come from the consumption module that defines poverty. They are legitimate (shares carry no level information), but such data are not available when targeting households that were never surveyed. Notebook 09 therefore also reports a "PMT-feasible" feature set without them.

## 6. VUP coverage among high-need households

High need = the top quintile of the distress score excluding its VUP component (`distress_quintile_excl_vup == 5`), as defined in notebook 07.

In [ ]:
cov_q = cm.weighted_rate_table(dev, "vup_any", "distress_quintile_excl_vup", "weight", [1, 2, 3, 4, 5])
cov_q.index = [f"Need quintile {i}" + (" (lowest)" if i == 1 else " (highest)" if i == 5 else "") for i in cov_q.index]
pov_q = cm.weighted_rate_table(dev, "poor", "distress_quintile_excl_vup", "pop_wt", [1, 2, 3, 4, 5])
display(cov_q.assign(poor_pct_people=100 * pov_q["mean"].to_numpy()))
fig, (ax,) = pl.figure("VUP reaches the highest-need quintile less than the middle ones",
                       "Share of households receiving any VUP component, by quintile of the distress score excluding VUP. Development set", width=10, height=3.9)
pl.dot_ci(ax, cov_q, 100, pl.SERIES[2])
ax.set_xlim(0, 100 * cov_q["ci_high"].max() * 1.2)
ax.set_xlabel("% of households with any VUP component", fontsize=9, color=pl.INK["secondary"])
path = pl.save(fig, FIG / "vup_coverage_by_need_quintile.png", top=0.72, bottom=0.2, left=0.25)
display(Image(str(path)))

In [ ]:
hn = dev[dev["distress_quintile_excl_vup"].eq(5)].copy()
print(f"High-need households in the development set: {len(hn):,} ({hn['clust'].nunique()} clusters); VUP recipients: {int(hn['vup_any'].sum())}; "
      f"weighted coverage {100 * np.average(hn['vup_any'], weights=hn['weight']):.1f}%")
cov_hn = {k: cm.weighted_rate_table(hn, "vup_any", col, "weight", [g for g in order_ if (hn[col] == g).any()])
          for k, (col, order_) in {"area": ("area_name", ["Urban", "Rural"]),
                                   "income_source": ("income_source", DIMS["income_source"][2]),
                                   "poverty": ("poor", [1, 0])}.items()}
for k, t in cov_hn.items():
    print(k); display(t)
by_d = hn.groupby("district_name").apply(lambda g: pd.Series({"high-need households (sample)": len(g), "VUP recipients": int(g["vup_any"].sum()),
          "coverage %": 100 * np.average(g["vup_any"], weights=g["weight"]),
          "uncovered high-need (weighted households, development set)": g.loc[g["vup_any"].eq(0), "weight"].sum()})).sort_values("coverage %")
cov_d = cm.weighted_rate_table(hn, "vup_any", "district_name", "weight", by_d.index.tolist())
fig, (ax,) = pl.figure("VUP coverage of high-need households by district",
                       "Highest need quintile only; sorted by coverage. Small samples: most intervals overlap. Development set",
                       width=10, height=1.9 + 0.3 * len(cov_d))
pl.dot_ci(ax, cov_d, 100, pl.SERIES[2], reference=np.average(hn["vup_any"], weights=hn["weight"]))
ax.set_xlim(0, 100 * cov_d["ci_high"].max() * 1.12)
ax.set_xlabel("% of high-need households with any VUP component", fontsize=9, color=pl.INK["secondary"])
path = pl.save(fig, FIG / "vup_coverage_high_need_by_district.png", top=1 - 1.25 / (1.9 + 0.3 * len(cov_d)), bottom=0.6 / (1.9 + 0.3 * len(cov_d)), left=0.25)
display(Image(str(path)))
display(by_d)

In [ ]:
# How stable is the high-need group to the definition?
full_q = df.merge(splits, on=["hhid", "clust"])
w_all = full_q["weight"]
from src.features.household_cpi_shock import weighted_quintile
pf_q = pd.Series(weighted_quintile(full_q["need_score_pf"], w_all), index=full_q.index)
orig = full_q["distress_quintile_excl_vup"].eq(5)
alt = pf_q.eq(5)
agree = (orig == alt).mean()
print(f"Top-quintile membership agreement, published score vs program-free score: {100 * agree:.1f}% of households; "
      f"{int((orig & ~alt).sum())} leave and {int((alt & ~orig).sum())} enter the group (all splits)")
print(f"Of households leaving: {int((orig & ~alt & full_q['vup_any'].eq(1)).sum())} are VUP recipients; of households entering: {int((alt & ~orig & full_q['vup_any'].eq(1)).sum())}")

## 7. Save

In [ ]:
group_summary.to_csv(ROOT / "reports" / "figures" / "modeling_eda" / "poverty_distress_by_group.csv", index=False)
print("Charts:", ", ".join(sorted(p.name for p in FIG.glob("*.png"))))
print(f"Splits saved to {cm.SPLITS_PATH.relative_to(ROOT)}")